# Predictive Analytics: Support Vector Machines

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [14]:
from run_config import PATHS

In [16]:
import pandas as pd
import numpy as np
import datetime
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.utils import resample
from sklearn.base import clone 

## Preparations

In [17]:
INPUT = PATHS.train_test_dir

In [ ]:
#GRID_SAMPLE = 35_000 # full or number
SPATIAL_UNIT = "COMMUNITY_AREAS" # HEXAGON
TIME_UNIT = "24H" # options: 1H, 4H, 24H
H3_RES = "7" # options 7,8

In [19]:
# "Settings" / Decisions for the training data

DATA_PATH_TRAIN = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TRAIN.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TEST.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_VAL.parquet"
if SPATIAL_UNIT == "HEXAGON":
    DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
    DATA_PATH_TRAIN = INPUT / F"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
    DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"

MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_demand",
    "date",
]

Load data and select features and target

In [20]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
val_df = pd.read_parquet(DATA_PATH_VAL)
test_df = pd.read_parquet(DATA_PATH_TEST)

In [21]:
val_df = val_df.sample(n=min(5_000, len(val_df)), random_state=42)

In [22]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

# Community_area is a categorical id, not a numeric quantity, so one-hot encode it
X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])

# Keep the dummy columns before scaling turns X_train into a plain array
train_columns = X_train.columns

# Make sure val/test have the same dummy columns as train (in case a community_area is missing)
X_val = X_val.reindex(columns=train_columns, fill_value=0)
X_test = X_test.reindex(columns=train_columns, fill_value=0)


y_train = train_df[TARGET_COL]
y_val = val_df[TARGET_COL]
y_test = test_df[TARGET_COL]

# SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Features:", feature_cols)
print("Target:", y_train.dtypes)

Features: ['month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'hour_sin', 'hour_cos', 'is_holiday', 'community_area', 'weather_station_distance_km', 'food_drink', 'landmark', 'shop', 'train_station', 'tmpc', 'relh', 'sknt', 'p01m', 'vsby', 'wind_dir_sin', 'wind_dir_cos', 'station_observed', 'weather_imputed', 'precipitation_missing', 'weather_rain', 'weather_snow', 'weather_fog_mist', 'weather_thunder', 'weather_freezing', 'precipitation_trace', 'weather_qc_corrected', 'skyc1_CLR', 'skyc1_FEW', 'skyc1_SCT', 'skyc1_BKN', 'skyc1_OVC', 'skyc1_VV', 'weather_station_MDW', 'weather_station_ORD', 'weather_station_IGQ']
Target: uint32


In [23]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2026-04-23 08:00:00,4,4,8,1.000000e+00,6.123234e-17,0.433884,-0.900969,8.660254e-01,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00,0.00,No trips
1,2025-03-09 00:00:00,3,7,0,8.660254e-01,5.000000e-01,-0.781831,0.623490,0.000000e+00,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00,0.00,No trips
2,2025-05-30 16:00:00,5,5,16,8.660254e-01,-5.000000e-01,-0.433884,-0.900969,-8.660254e-01,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00,0.00,No trips
3,2025-06-30 20:00:00,6,1,20,5.000000e-01,-8.660254e-01,0.000000,1.000000,-8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00,0.00,No trips
4,2025-03-09 12:00:00,3,7,12,8.660254e-01,5.000000e-01,-0.781831,0.623490,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00,0.00,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156613,2025-07-16 12:00:00,7,3,12,1.224647e-16,-1.000000e+00,0.974928,-0.222521,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,88.5,22.125,12.75,28.75,Prcard
156614,2025-04-20 04:00:00,4,7,4,1.000000e+00,6.123234e-17,-0.781831,0.623490,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00,0.00,No trips
156615,2025-10-29 20:00:00,10,3,20,-1.000000e+00,-1.836970e-16,0.974928,-0.222521,-8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00,0.00,No trips
156616,2025-07-16 00:00:00,7,3,0,1.224647e-16,-1.000000e+00,0.974928,-0.222521,0.000000e+00,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00,0.00,No trips


In [24]:
model = SVR()

In [25]:
X_val = X_val

In [ ]:
param_grid_linear = {
    "C": [1, 10],
    "epsilon": [0.1, 0.5, 1],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [1, 10],
    "epsilon": [0.1, 0.5],
    "kernel": ["rbf", "sigmoid"],
    "gamma": [0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "epsilon": [0.01, 0.1, 0.5, 1],
    "kernel": ["poly"],
    "degree": [3, 4, 5],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVR(),
        param_grid=grid,
        cv=3,
        scoring="r2",
        n_jobs=1,
        error_score="raise"
    )
    search.fit(X_val, y_val)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.4595937099790876 best params: {'C': 10, 'epsilon': 1, 'kernel': 'linear'}
rbf_sigmoid best score: 0.29167800750717915 best params: {'C': 10, 'epsilon': 0.5, 'gamma': 0.01, 'kernel': 'rbf'}
poly best score: 0.5475672724432878 best params: {'C': 10, 'degree': 5, 'epsilon': 1, 'gamma': 0.01, 'kernel': 'poly'}
Overall best: poly {'C': 10, 'degree': 5, 'epsilon': 1, 'gamma': 0.01, 'kernel': 'poly'}


In [27]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 10, 'degree': 5, 'epsilon': 1, 'gamma': 0.01, 'kernel': 'poly'}
Best CV score: 0.5475672724432878


In [ ]:
# Train SVR with the best hyperparameters found by grid search
best_params = grid_search.best_params_
model = SVR(**best_params)
model.fit(X_train, y_train)

In [ ]:
# Make prediction 
y_pred = model.predict(X_test)

In [ ]:
y_pred

array([-1.31339049,  0.81902821,  1.97237673, ..., -0.64759251,
       -2.55128149, -0.49586741], shape=(223531,))

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 5.5602808155517005
MSE: 298.15442265439026
RMSE: 17.267148654435978
R2 Score: 0.7588652846925792


In [ ]:
# testing out at which point the code breaks
for n in [50_000, 70_000, 100_000, 150_000, 200_000, 500_000, len(X_train)]:
    X_sub, y_sub = resample(X_train, y_train, n_samples=n, random_state=42)
    m = clone(model)
    m.set_params(regressor__svm__max_iter=100_000)
    m.fit(X_sub, y_sub)
    pred = m.predict(X_test)
    print(n, r2_score(y_test, pred))